# 03 - Machine Learning Fundamentals with Scikit-learn

Train a simple classifier to identify stressed observations from financial indicators.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from analysis_utils import ensure_output_dir, load_indicator_data

ENV_FILE = ".env"  # Windows learners can change this to ".env.windows"
output_dir = ensure_output_dir()
df = load_indicator_data(ENV_FILE)
df["StressFlag"].value_counts(normalize=True)

## Prepare Features and Target

In [ ]:
features = [
    "LiquidityRatio",
    "CapitalAdequacyRatio",
    "NplRatio",
    "TransactionVolume",
    "TransactionValueLSL",
    "CreditGrowthRate",
    "InflationRate",
    "InterbankRate",
]

X = df[features]
y = df["StressFlag"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

X_train.shape, X_test.shape

## Train Logistic Regression Model

In [ ]:
model = Pipeline([
    ("scale", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced")),
])

model.fit(X_train, y_train)
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"Accuracy: {accuracy:.3f}")
print(classification_report(y_test, predictions))

## Confusion Matrix

In [ ]:
matrix = confusion_matrix(y_test, predictions)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
ax.set_title("Stress Classifier Confusion Matrix")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_xticklabels(["Normal", "Stress"])
ax.set_yticklabels(["Normal", "Stress"], rotation=0)
fig.tight_layout()
fig.savefig(output_dir / "stress_classifier_confusion_matrix.png", dpi=160)
plt.show()

## Feature Coefficients

In [ ]:
coefficients = pd.DataFrame({
    "feature": features,
    "coefficient": model.named_steps["classifier"].coef_[0],
}).sort_values("coefficient", ascending=False)

metrics = pd.DataFrame([{
    "model": "LogisticRegression",
    "accuracy": accuracy,
    "train_rows": len(X_train),
    "test_rows": len(X_test),
}])

coefficients.to_csv(output_dir / "stress_model_coefficients.csv", index=False)
metrics.to_csv(output_dir / "model_metrics.csv", index=False)
coefficients